# Session 3: Tree Ensembles and Explaining Black-Box Regressors

## Overview

A ready-mix concrete supplier wants to predict the **compressive strength** of a mix from its recipe and its
age, so it can stop crushing so many test cylinders. Unlike the power plant, the relationships here are strongly
non-linear and interact: strength climbs quickly in the first weeks and then levels off, and how much water hurts
depends on how much cement there is. A linear model struggles. **Random forests** and **gradient boosting** find
these shapes on their own, and cut the error in half.

The price is that nobody can read a forest of 500 trees. The second half of the session shows how to explain a
black-box model anyway: which inputs matter (**permutation importance**), how the prediction changes with each
input (**partial dependence**), and why one particular mix got its prediction (**SHAP values**).

By the end, you will be able to:

- Explain how a decision tree, a random forest and gradient boosting make predictions.
- Compare linear and tree-based regressors fairly with cross-validation.
- Measure global input importance with permutation importance.
- Read partial dependence and ICE plots to see the shape a model has learned.
- Explain a single prediction with SHAP values, and check that the explanation adds up.

**Prerequisites:** Session 2 (pipelines, cross-validation).

**Dataset:** [UCI Concrete Compressive Strength](https://archive.ics.uci.edu/dataset/165/concrete+compressive+strength).
1,030 laboratory tests (Yeh, 1998): the amount of each of seven ingredients in kg per m³ of concrete, the age of
the sample in days (1 to 365), and the measured compressive strength in megapascals (MPa). It is downloaded
automatically by the code.

## Where this fits

| Part | Sessions | Task | Algorithms |
| --- | --- | --- | --- |
| 1. Regression | 1, 2, 3, 4 | Predict a number | Linear regression by gradient descent and in scikit-learn, tree ensembles, explaining black-box models **← you are here** |
| 2. Classification | 5, 6, 7, 8, 9 | Predict a category | Logistic regression, naive Bayes, linear text classifiers, comparing classifiers |
| 3. Clustering | 10, 11 | Find groups without labels | k-means, hierarchical clustering |
| 4. Topic modelling | 12 | Find themes in text | LDA, NMF |

**This session's question:** Can a flexible model predict concrete strength much better than a linear one, and
can we still explain what it does?

## Step 1: Set up the environment

This module has one pinned `requirements.txt` (in the module folder) for all twelve sessions. If you have
not done it yet, create a virtual environment and install it once, from the module folder:

```bash
python3.11 -m venv .venv
source .venv/bin/activate          # Windows: .venv\Scripts\Activate.ps1
pip install -r requirements.txt
jupyter notebook
```

This session uses:

| Package | Why we need it |
| --- | --- |
| ucimlrepo | Downloads the dataset from UCI |
| scikit-learn | Linear model, trees, forests, boosting, permutation importance, partial dependence |
| shap | Explanations of individual predictions |
| pandas | Tables |
| numpy | Arrays |
| matplotlib | Charts |

Run the cell below. It imports what this notebook needs and prints the versions. The numbers quoted in
this notebook come from these exact versions; a different version can shift the last decimal place.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")    # harmless notice from shap's progress bar
import shap
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.tree import DecisionTreeRegressor, export_text

from importlib.metadata import version

for package in ["ucimlrepo", "scikit-learn", "shap", "pandas", "numpy", "matplotlib"]:
    print(f"{package:13s} {version(package)}")

## Step 2: Load the data, remove duplicates, split

The file contains some **exact duplicate rows**: the same recipe, age and strength recorded twice. If one copy
lands in the training set and the other in the test set, the model is tested on a row it has already seen, and
the score looks better than it really is. We remove them *before* splitting.

In [ ]:
from ucimlrepo import fetch_ucirepo

concrete = fetch_ucirepo(id=165).data.original
concrete.columns = ["cement", "slag", "fly_ash", "water", "superplasticizer",
                    "coarse_agg", "fine_agg", "age", "strength"]
print(f"{len(concrete):,} rows, {concrete.duplicated().sum()} exact duplicates")
concrete = concrete.drop_duplicates().reset_index(drop=True).astype(float)

FEATURES = concrete.columns.drop("strength").tolist()
train, test = train_test_split(concrete, test_size=0.2, random_state=42)
X_train, y_train = train[FEATURES], train["strength"]
X_test, y_test = test[FEATURES], test["strength"]
print(f"{len(train)} mixes to train on, {len(test)} held out")
print(train.describe().round(1).T[["mean", "min", "max"]])

**Read the output:** 25 duplicates removed, leaving 1,005 tests: 804 for training and 201 held out. Ingredients
are in kg per m³. Slag, fly ash and superplasticizer are often zero (many mixes do not use them). Strength in the
training set ranges from 3.3 to 80 MPa. `astype(float)` stores every column as a decimal number; some tools used
later (partial dependence) warn about whole-number columns such as age.

## Step 3: Look at two key relationships

Concrete gains strength as it cures, and more cement generally means stronger concrete. Plot both.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(train["age"], train["strength"], s=8, alpha=0.4)
axes[0].set_xlabel("age (days)")
axes[0].set_ylabel("strength (MPa)")
axes[0].set_title("Strength rises fast, then levels off")
axes[1].scatter(train["cement"], train["strength"], s=8, alpha=0.4, c=np.log(train["age"]), cmap="viridis")
axes[1].set_xlabel("cement (kg/m³)")
axes[1].set_title("More cement, stronger (colour = log age)")
plt.show()

print(train.groupby("age")["strength"].agg(["count", "mean"]).round(1).T)

**Read the output:** age only takes a handful of values (3, 7, 14, 28, 56, 90 days and so on): these are the
standard test days. Mean strength rises steeply, from 18.4 MPa at 3 days to 35.8 at 28 days and 51.3 at 56 days.
Beyond that the averages jump around (40.9 at 90 days, 67.8 at 91) because different recipes were tested at
different ages, but the curve clearly flattens. No straight line in age can follow it. Cement shows a clear but noisy upward trend, and the colours show that at the same
cement content, older samples are stronger: inputs **interact**.

## Step 4: The linear baseline

Start with Session 2's tool, scored by 5-fold cross-validation on the training set.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=0)


def cv_rmse(model, X=X_train):
    return -cross_val_score(model, X, y_train, cv=cv, scoring="neg_root_mean_squared_error").mean()


print(f"predict the average: CV RMSE = {y_train.std(ddof=0):5.2f} MPa")
print(f"linear regression:   CV RMSE = {cv_rmse(LinearRegression()):5.2f} MPa")


def add_domain_features(X):
    return X.assign(water_cement=X["water"] / X["cement"], log_age=np.log(X["age"]))


print(f"linear + water/cement ratio and log(age): CV RMSE = "
      f"{cv_rmse(LinearRegression(), add_domain_features(X_train)):5.2f} MPa")

**Read the output:** predicting the average misses by 16.0 MPa (the standard deviation of strength). A linear
model brings this down to 10.1 MPa. Adding two features a concrete engineer would suggest, the **water/cement
ratio** (the classic rule of thumb for strength) and the **log of the age** (to model the levelling-off), improves
it to 6.8 MPa.

That is Session 2's lesson again: a linear model is only as good as the features you give it. Here we knew
which features to invent. A tree-based model can find this kind of shape by itself.

## Step 5: A single decision tree

A **decision tree** splits the data with a sequence of yes/no questions ("is age ≤ 21 days?"), choosing at each
step the question that best separates high from low strength. Each final group (a *leaf*) predicts the average
strength of its training mixes. Print a depth-2 tree to see the questions it asks.

In [ ]:
small_tree = DecisionTreeRegressor(max_depth=2, random_state=0).fit(X_train, y_train)
print(export_text(small_tree, feature_names=FEATURES, decimals=1))

for depth in [2, 5, 10, None]:
    tree = DecisionTreeRegressor(max_depth=depth, random_state=0)
    print(f"tree, max_depth={str(depth):4s}  CV RMSE = {cv_rmse(tree):5.2f} MPa")

**Read the output:** the first question is "age ≤ 21 days?", then "cement ≤ about 353 kg?": the tree discovered on
its own that young samples are weak and that cement drives strength. Its four leaves predict from 18.9 MPa (young,
less cement) to 54.7 MPa (older, more cement). With only 4 possible predictions, its error is large (11.9 MPa).

A deeper tree asks more questions and fits finer detail: depth 10 reaches 7.1 MPa with no feature engineering,
far better than the plain linear model (10.1). An unlimited tree (`None`) keeps splitting until every leaf is
nearly pure, and is slightly *worse* (7.4): it has started memorising the training mixes. A single deep tree is
also **unstable**: change a few training rows and the questions change. The next two models fix both problems by
combining many trees.

## Step 6: Random forest and gradient boosting

Two ways to combine trees:

- A **random forest** grows hundreds of deep trees, each on a different random sample of the rows (a *bootstrap*
  sample) and considering a random subset of the inputs at each split. It **averages** their predictions. Each tree
  overfits in its own way, and averaging cancels much of it out.
- **Gradient boosting** grows small trees **one after another**. Each new tree is fitted to the errors the trees
  so far still make, and its prediction is added in, scaled down by a **learning rate**. The model improves step by
  step, like gradient descent in Session 1, but each step is a tree.

In [ ]:
models = {
    "random forest": RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1),
    "gradient boosting": GradientBoostingRegressor(n_estimators=500, learning_rate=0.05, max_depth=4,
                                                   subsample=0.8, random_state=0),
}
for name, model in models.items():
    print(f"{name:18s} CV RMSE = {cv_rmse(model):5.2f} MPa")

**Read the output:** the random forest reaches 5.2 MPa and gradient boosting 4.5 MPa, with no feature
engineering: less than half the plain linear model's error (10.1), and well below the engineered one (6.8). Gradient boosting usually wins on tables of
data like this one, and it has more settings to tune (number of trees, learning rate, tree depth); the values
here are sensible defaults, not tuned.

## Step 7: Final check on the test set

Fit each model on the full training set and score it once on the 201 held-out mixes.

In [ ]:
linear = LinearRegression().fit(add_domain_features(X_train), y_train)
forest = models["random forest"].fit(X_train, y_train)
boost = models["gradient boosting"].fit(X_train, y_train)

predictions = {
    "linear + domain features": linear.predict(add_domain_features(X_test)),
    "random forest": forest.predict(X_test),
    "gradient boosting": boost.predict(X_test),
}
print(pd.DataFrame({name: {"MAE": mean_absolute_error(y_test, p),
                           "RMSE": mean_squared_error(y_test, p) ** 0.5,
                           "R²": r2_score(y_test, p)} for name, p in predictions.items()}).T.round(3))

**Read the output:** the test results agree with cross-validation. Gradient boosting predicts the strength of new
mixes to within 2.7 MPa on average (RMSE 4.3, R² 0.938). The engineered linear model's errors are about twice as
large (MAE 5.8), and the random forest sits in between.

Gradient boosting is the model we would use. Now the question every engineer will ask: **what has it learned, and
can we trust it?**

## Step 8: Which inputs matter? Permutation importance

**Permutation importance** asks: if we scramble one input's column in the test set (destroying its information
but keeping its values realistic), how much worse does the model get? A big increase in error means the model
relies on that input. It works for any model, because it only needs predictions.

In [ ]:
perm = permutation_importance(boost, X_test, y_test, scoring="neg_root_mean_squared_error",
                              n_repeats=20, random_state=0)
importance = pd.DataFrame({"RMSE increase (MPa)": perm.importances_mean, "spread": perm.importances_std},
                          index=FEATURES).sort_values("RMSE increase (MPa)", ascending=False)
print(importance.round(2))

importance["RMSE increase (MPa)"].iloc[::-1].plot.barh(figsize=(7, 3.5),
                                                       xerr=importance["spread"].iloc[::-1])
plt.xlabel("increase in test RMSE when the input is scrambled (MPa)")
plt.title("Permutation importance, gradient boosting")
plt.show()

**Read the output:** **age** and **cement** are by far the most important: scrambling them adds 11.1 and 9.7 MPa
to the error. Slag and water come next (about 4 MPa each), then superplasticizer. The aggregates and fly ash
matter least (fly ash only 0.1 MPa).
This matches what concrete engineers know, which is a good sign: the model has learned real physics, not a
quirk of the data.

Two cautions. Importance is about **this model**, not about the world: an input the model happens not to use can
still matter. And when two inputs are strongly correlated, scrambling one can look harmless because the model
gets the same information from the other.

## Step 9: How does the prediction change? Partial dependence

Importance says *how much* an input matters, not *how*. A **partial dependence plot** (PDP) shows the average
prediction as one input is varied across its range, with the other inputs left as they are in the data. The thin
lines (**ICE**, individual conditional expectation) show the same curve for individual mixes; when they are not
parallel, the input interacts with others.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
PartialDependenceDisplay.from_estimator(boost, X_train, ["age", "cement", "water"], kind="both",
                                        subsample=60, random_state=0, ax=axes,
                                        ice_lines_kw={"alpha": 0.2}, pd_line_kw={"color": "tab:red"})
plt.suptitle("Partial dependence (red) and ICE curves (blue)")
plt.show()

**Read the output:**

- **age:** a steep rise over the first month, then a plateau, exactly the curing curve from Step 3, learned with
  no `log(age)` feature.
- **cement:** close to a straight upward line: more cement, more strength.
- **water:** strength falls as water rises, the water/cement rule, again discovered from the data. (The curve
  turns up slightly at the very highest water contents, where there are few mixes; do not read much into the
  edges of a PDP.)

The ICE lines for age fan out: the extra strength gained with age differs between mixes (it depends on the
recipe). That is an interaction a plain linear model cannot represent without help.

## Step 10: Why this prediction? SHAP values

Global explanations describe the model on average. Often we need to explain **one** prediction: "why does the
model think this mix will be weak?". **SHAP values** split one prediction into contributions, one per input:

$$\text{prediction} = \text{average prediction} + \sum_{\text{inputs}} \text{SHAP value of that input}$$

A positive SHAP value pushed this mix's prediction up, a negative one pushed it down. `shap.TreeExplainer` computes
them exactly and quickly for tree models.

In [ ]:
explainer = shap.TreeExplainer(boost)
explanation = explainer(X_test)

i = int(np.argmin(boost.predict(X_test)))          # the mix the model thinks is weakest
print(f"mix {X_test.index[i]}:  predicted {boost.predict(X_test.iloc[[i]])[0]:.1f} MPa, "
      f"measured {y_test.iloc[i]:.1f} MPa")
print(X_test.iloc[i].to_string())
print(f"\nbase value {explanation.base_values[i]:.2f} + sum of SHAP values "
      f"{explanation.values[i].sum():.2f} = {explanation.base_values[i] + explanation.values[i].sum():.2f}")

shap.plots.waterfall(explanation[i], show=False)
plt.title("Why the model predicts this mix will be weak")
plt.show()

**Read the output:** the model's weakest prediction (3.8 MPa) is a 3-day-old sample with a below-average 255 kg of
cement and no slag, fly ash or superplasticizer. The waterfall starts from the **base value** (the average
prediction, 35.0 MPa) and adds each input's contribution: the young **age** pulls the prediction down by about
15 MPa, by far the most; the large amount of **fine aggregate** (945 kg, against a typical 773) takes off about
5 more; water, cement, the missing superplasticizer and the missing slag take about 2-2.5 MPa each. The
contributions add up exactly to the prediction: 35.02 - 31.21 = 3.81, as the printed check confirms.

This is the explanation you would give a customer: "this sample is weak mainly because it is only three days old;
the recipe, sandy and with no strength-boosting additives, makes it weaker still."

Note that the sample actually measured 8.2 MPa: the model was too pessimistic by 4 MPa. SHAP explains **what the
model did**, not what is true.

## Step 11: All explanations at once

A **beeswarm** plot shows the SHAP values for every test mix: one dot per mix per input, coloured by the input's
value (red = high, blue = low).

In [ ]:
shap.plots.beeswarm(explanation, show=False)
plt.title("SHAP values for all 201 test mixes")
plt.show()

**Read the output:** inputs are sorted by overall impact (mean absolute SHAP value), and the order broadly agrees
with permutation importance: age and cement on top, then water and slag, with fly ash last. The colours give the direction: for **age**, blue dots (young samples) sit far to the left
(much weaker), while red dots (old samples) sit to the right; for **water**, red dots (lots of water) sit on the
left. One plot summarises both *how much* and *which way*.

## What this session hands to the next

- **Tree ensembles** as the strong default for tabular data: they find non-linear shapes and interactions without
  feature engineering, and gradient boosting usually wins.
- **Three explanation tools that work on any model:** permutation importance (which inputs), partial dependence
  and ICE (how), SHAP (why, for one prediction).
- **A habit:** check that the explanations agree with domain knowledge before trusting the model. Session 4 puts
  everything from Sessions 1-3 together in an end-to-end project.

## Common mistakes and fixes

| Mistake | What goes wrong | Fix |
| --- | --- | --- |
| Leaving duplicate rows in the data | The same row in training and test; scores look too good | `drop_duplicates()` before splitting |
| Using one deep decision tree | Unstable; small data changes give a different tree | Use a random forest or gradient boosting |
| Reading the forest's built-in `feature_importances_` as the truth | Biased towards inputs with many distinct values | Use permutation importance on held-out data |
| Treating importance as cause and effect | The model can rely on a proxy for the real cause | Combine with domain knowledge; experiments establish causes |
| Reading a PDP when inputs are strongly correlated | The plot averages over impossible combinations | Check ICE lines and correlations; prefer SHAP |
| Gradient boosting with a high learning rate and many trees | Overfits | Lower the learning rate, choose the number of trees by CV |
| Forgetting `show=False` with SHAP plots | Title and other changes are not applied | Pass `show=False`, then `plt.show()` |

## Exercises

- [ ] Fit the random forest with `max_features=1.0` (each split may consider every input). What happens to the CV
  RMSE? Why does restricting the inputs at each split help?
- [ ] Plot the gradient boosting CV RMSE for `n_estimators` in 50, 100, 250, 500, 1000 at learning rates 0.05 and 0.2.
- [ ] Explain the mix the model thinks is *strongest* (`np.argmax`). Which inputs push it up?
- [ ] Draw a 2-D partial dependence plot for `("cement", "water")` (pass the pair as one feature in the list).
  Describe the interaction.

Next, Session 4 is a bootcamp: an end-to-end regression project on bike-sharing demand, from a first look at the
data to a model you can defend.